# Lab 2: Computation Graphs and Forward Propagation

## 🎯 Learning Objectives

By the end of this lab, you will understand:
- What computation graphs are and why they matter
- How to build a Value class that tracks operations
- Forward propagation through computation graphs
- NumPy fundamentals for efficient numeric computation
- How to visualize computation graphs

**Why This Matters:** Every deep learning framework (PyTorch, TensorFlow) uses computation graphs to track operations. This enables automatic differentiation, which we'll implement in later labs.

**Note:** You can now use NumPy! You've earned it after understanding the internals in Lab 1.

## Part 1: What is a Computation Graph?

### The Big Idea

When you compute `d = (a + b) * c`, you're actually creating a **graph** of operations:

```
  a    b        Inputs
   \  /
    (+)    →  e   Intermediate result
     |  \ 
     |   c      Another input
     \  /
      (*)   →  d   Final output
```

This graph records:
1. **What values** were involved (a, b, c, e, d)
2. **What operations** were performed (+, *)
3. **How they connect** (parent-child relationships)

### Why This Matters

In deep learning:
- **Forward pass:** Compute outputs from inputs (what we do now)
- **Backward pass:** Compute gradients from outputs to inputs (next lab)

The computation graph is the **recipe** that lets us do both!

### Example: A Simple Computation Graph

Let's trace through `f = (a + b) * c` step by step:

```python
a = 2.0
b = 3.0
c = 4.0

# Step 1: a + b = 5.0
e = a + b  # e remembers it came from a and b via +

# Step 2: e * c = 20.0
f = e * c  # f remembers it came from e and c via *
```

The graph looks like:
```
a=2.0 ──┐
        ├──(+)──> e=5.0 ──┐
b=3.0 ──┘                 ├──(*)──> f=20.0
        c=4.0 ────────────┘
```

## Part 2: Building the Value Class

### Design Goals

Our `Value` class needs to:
1. **Store the data** (the actual number; for example `e=5.0`)
2. **Remember parents** (which Values it came from; for example, parents of `e` are `a` and `b`)
3. **Remember operation** (what operation created it; for example, the operator for `e` is `+`)
4. **Support operators** (`+`, `-`, `*`, `/`, `sigmoid`, etc)

### The Value Class Structure

```python
class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = float(data)        # The actual number
        self.grad = 0.0                 # Gradient (used in backward pass)
        self._prev = set(_children)     # Parent nodes in the graph
        self._op = _op                  # Operation that created this node
        self.label = label              # Optional name for visualization
```

### Demo: How It Works

In [ ]:
# Simple demo of the Value class
class ValueDemo:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = float(data)
        self.grad = 0.0
        self._prev = set(_children)
        self._op = _op
        self.label = label
    
    def __add__(self, other):
        other = other if isinstance(other, ValueDemo) else ValueDemo(other)
        out = ValueDemo(self.data + other.data, (self, other), '+')
        return out
    
    def __repr__(self):
        return f"Value(data={self.data:.4f})"

# Test it
a = ValueDemo(2.0, label='a')
b = ValueDemo(3.0, label='b')
c = a + b
c.label = 'c'

print(f"a = {a}")
print(f"b = {b}")
print(f"c = a + b = {c}")
print(f"\nc._op = '{c._op}'  (operation that created c)")
print(f"c._prev = {c._prev}  (parents of c)")
print(f"\n✓ The computation graph is being recorded!")

## Exercise 1: Implement the Value Class

### Your Task

Implement a complete `Value` class that:
1. Tracks data and computation history
2. Supports arithmetic operators (+, -, *, /)
3. Handles operations with plain numbers (e.g., `Value(5) + 3`)

### Starter Code

In [ ]:
class Value:
    """A Value stores a scalar and tracks its computation history."""
    
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = float(data)
        self.grad = 0.0
        self._prev = set(_children)
        self._op = _op
        self.label = label
    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        return out
    
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        return out
    
    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supporting int/float powers"
        out = Value(self.data ** other, (self,), f'**{other}')
        return out
    
    def __neg__(self):
        return self * -1
    
    def __sub__(self, other):
        return self + (-other)
    
    def __truediv__(self, other):
        return self * (other ** -1)
    
    def __radd__(self, other):
        return self + other
    
    def __rmul__(self, other):
        return self * other
    
    def __repr__(self):
        return f"Value(data={self.data:.4f})"

# Test will validate

### Test Your Implementation

In [ ]:
print("=" * 60)
print("Test 1: Basic Operations")
print("=" * 60)

a = Value(2.0, label='a')
b = Value(3.0, label='b')
c = a + b
c.label = 'c'

print(f"a = {a}")
print(f"b = {b}")
print(f"c = a + b = {c}")
assert c.data == 5.0, f"Expected 5.0, got {c.data}"
assert c._op == '+', f"Expected '+', got {c._op}"
assert a in c._prev and b in c._prev, "c should have a and b as parents"
print("✓ Addition: PASS\n")

In [ ]:
print("=" * 60)
print("Test 2: Complex Expression")
print("=" * 60)

# f = (a + b) * c
a = Value(2.0, label='a')
b = Value(3.0, label='b')
c = Value(4.0, label='c')

e = a + b
e.label = 'e'
f = e * c
f.label = 'f'

print(f"f = (a + b) * c = ({a.data} + {b.data}) * {c.data} = {f.data}")
assert f.data == 20.0, f"Expected 20.0, got {f.data}"
print("✓ Complex expression: PASS\n")

In [ ]:
print("=" * 60)
print("Test 3: Operations with Plain Numbers")
print("=" * 60)

x = Value(5.0)
y = x + 3  # Should work even though 3 is not a Value
z = 2 * x  # Should work with reversed operands

print(f"x = {x}")
print(f"x + 3 = {y}")
print(f"2 * x = {z}")

assert y.data == 8.0, f"Expected 8.0, got {y.data}"
assert z.data == 10.0, f"Expected 10.0, got {z.data}"
print("✓ Mixed operations: PASS\n")

In [ ]:
print("=" * 60)
print("Test 4: All Operators")
print("=" * 60)

a = Value(10.0)
b = Value(3.0)

print(f"a + b = {a + b}")
print(f"a - b = {a - b}")
print(f"a * b = {a * b}")
print(f"a / b = {a / b}")
print(f"a ** 2 = {a ** 2}")

assert (a + b).data == 13.0
assert (a - b).data == 7.0
assert (a * b).data == 30.0
assert abs((a / b).data - 3.333333) < 0.001
assert (a ** 2).data == 100.0

print("\n✓ All operators: PASS\n")

In [ ]:
print("=" * 60)
print("🎉 All Tests Passed!")
print("=" * 60)
print("\nYour Value class can track computation graphs!")

## Part 3: Visualizing Computation Graphs

### Why Visualization Matters

Seeing the computation graph helps you:
- Understand how operations connect
- Debug issues in complex expressions
- Verify your implementation is correct

### Using the Visualization Helper

We provide a `visualize_graph` function (from Karpathy's micrograd) that draws the computation graph using Graphviz.

In [ ]:
from utils import visualize_graph

# Build a computation graph
a = Value(2.0, label='a')
b = Value(-3.0, label='b')
c = Value(10.0, label='c')

e = a * b
e.label = 'e'
d = e + c
d.label = 'd'

# Visualize it!
visualize_graph(d, title="Computation Graph: d = (a * b) + c")

**What you should see:**
- Nodes showing values (a=2.0, b=-3.0, c=10.0, e=-6.0, d=4.0)
- Operations (* and +) connecting them
- Arrows showing the flow from inputs to output
- Gradients are all 0.0 (we haven't computed them yet)

**Try visualizing your own expressions!**

In [ ]:
# Try a more complex expression
x = Value(2.0, label='x')
y = Value(3.0, label='y')
z = x ** 2 + y ** 2
z.label = 'z'

visualize_graph(z, title="Computation Graph: z = x² + y²")

## Part 3.5: Introduction to Gradients

### What Are Gradients?

A **gradient** tells us **how much the output changes** when we nudge an input slightly.

Think of it as **sensitivity**:
- If `output.grad = 5`, then increasing input by 0.001 increases output by ≈0.005
- Large gradient = output is very sensitive to this input
- Small gradient = output barely cares about this input

### Why Gradients Matter for Deep Learning

In neural networks:
- **Forward pass:** Compute predictions from inputs
- **Backward pass:** Compute gradients to know which direction to adjust weights
- **Optimization:** Use gradients to update weights and reduce error

**The computation graph we built enables automatic gradient calculation!**

### Simple Example: Linear Function

For `f(x) = 2x`, the gradient is always 2:

```
f(3) = 6
f(3.001) = 6.002
Change = 6.002 - 6.000 = 0.002 = 2 × 0.001
Gradient = 0.002 / 0.001 = 2 ✓
```

In [ ]:
# Demo: Computing gradient numerically
def f(x):
    return 2 * x

x = 3.0
h = 0.001  # Small nudge

# Numerical gradient
f_x = f(x)
f_x_plus_h = f(x + h)
gradient = (f_x_plus_h - f_x) / h

print("=" * 60)
print("Numerical Gradient Demo")
print("=" * 60)
print(f"f(x) = 2x")
print(f"f({x}) = {f_x}")
print(f"f({x + h}) = {f_x_plus_h}")
print(f"Gradient = (f(x+h) - f(x)) / h = {gradient:.6f}")
print(f"Expected: 2.0 ✓")

### Gradients in Computation Graphs

For complex expressions like `f = (a + b) * c`, we need gradients of f with respect to each input:
- `df/da` = How does f change when a changes?
- `df/db` = How does f change when b changes?
- `df/dc` = How does f change when c changes?

**The Chain Rule** lets us compute these by traversing the computation graph backwards!

### Example: Computing Gradients

For `f = (a + b) * c` where a=2, b=3, c=4:

**Forward pass:**
```
e = a + b = 5
f = e * c = 20
```

**Backward pass (gradients):**
```
df/df = 1  (by definition)
df/dc = e = 5  (because f = e * c, so ∂(e*c)/∂c = e)
df/de = c = 4  (because f = e * c, so ∂(e*c)/∂e = c)
df/da = df/de * de/da = 4 * 1 = 4  (chain rule!)
df/db = df/de * de/db = 4 * 1 = 4  (chain rule!)
```

Let's verify this numerically:

In [ ]:
# Verify gradients numerically
def compute_f(a_val, b_val, c_val):
    e = a_val + b_val
    f = e * c_val
    return f

a, b, c = 2.0, 3.0, 4.0
h = 0.0001

f_original = compute_f(a, b, c)

# Gradient with respect to a
f_a_nudged = compute_f(a + h, b, c)
grad_a = (f_a_nudged - f_original) / h

# Gradient with respect to b
f_b_nudged = compute_f(a, b + h, c)
grad_b = (f_b_nudged - f_original) / h

# Gradient with respect to c
f_c_nudged = compute_f(a, b, c + h)
grad_c = (f_c_nudged - f_original) / h

print("=" * 60)
print("Numerical Gradient Verification")
print("=" * 60)
print(f"f = (a + b) * c = ({a} + {b}) * {c} = {f_original}")
print(f"\nNumerical gradients:")
print(f"df/da ≈ {grad_a:.4f}  (expected: 4.0)")
print(f"df/db ≈ {grad_b:.4f}  (expected: 4.0)")
print(f"df/dc ≈ {grad_c:.4f}  (expected: 5.0)")
print(f"\n✓ Matches our analytical calculation!")

### Key Takeaways About Gradients

**1. Gradients measure sensitivity**
- How much does the output change for a small change in input?

**2. The computation graph enables gradient calculation**
- Forward pass: compute outputs
- Backward pass: compute gradients (next lab!)

**3. Chain rule connects gradients**
- For nested expressions, multiply gradients along the path
- Example: `df/da = (df/de) × (de/da)`

**4. Numerical vs Analytical gradients**
- Numerical: approximate by nudging (slow, but verifies correctness)
- Analytical: exact math (fast, what we'll implement!)

### Coming Up in Lab 3

In the next lab, you'll implement **backward propagation** to automatically compute all gradients using the chain rule. This is the foundation of training neural networks!

For now, just remember: **the computation graph we built in this lab makes automatic differentiation possible**.

## Part 4: Introduction to NumPy

### You've Earned NumPy!

After implementing array operations from scratch in Lab 1, you now understand **how** NumPy works internally. Time to use the real deal!

### NumPy Basics

NumPy is the foundation of scientific computing in Python. It provides:
- Fast array operations (implemented in C)
- Broadcasting for flexible shape handling
- Linear algebra operations

**Documentation:** https://numpy.org/doc/stable/

### Creating Arrays

In [ ]:
import numpy as np

print("=" * 60)
print("Creating NumPy Arrays")
print("=" * 60)

# From Python list
a = np.array([1, 2, 3, 4])
print(f"1D array: {a}")
print(f"Shape: {a.shape}")
print(f"Type: {a.dtype}\n")

# 2D array (matrix)
b = np.array([[1, 2, 3], [4, 5, 6]])
print(f"2D array:\n{b}")
print(f"Shape: {b.shape}  (2 rows, 3 columns)\n")

# Special arrays
zeros = np.zeros((2, 3))  # All zeros
ones = np.ones((2, 3))    # All ones
identity = np.eye(3)       # Identity matrix

print(f"Zeros (2x3):\n{zeros}\n")
print(f"Ones (2x3):\n{ones}\n")
print(f"Identity (3x3):\n{identity}")

### Indexing and Slicing

In [ ]:
print("=" * 60)
print("Indexing and Slicing")
print("=" * 60)

x = np.array([[1, 2, 3, 4],
              [5, 6, 7, 8],
              [9, 10, 11, 12]])

print(f"Original array:\n{x}\n")

# Single element
print(f"x[0, 0] = {x[0, 0]}  (first element)")
print(f"x[1, 2] = {x[1, 2]}  (row 1, col 2)\n")

# Slicing rows
print(f"First row: x[0, :] = {x[0, :]}")
print(f"Last row: x[-1, :] = {x[-1, :]}\n")

# Slicing columns
print(f"First column: x[:, 0] = {x[:, 0]}")
print(f"Last column: x[:, -1] = {x[:, -1]}\n")

# Subarray
print(f"First 2 rows, first 3 cols:\n{x[:2, :3]}")

### Element-wise Operations

In [ ]:
print("=" * 60)
print("Element-wise Operations")
print("=" * 60)

a = np.array([1, 2, 3, 4])
b = np.array([10, 20, 30, 40])

print(f"a = {a}")
print(f"b = {b}\n")

print(f"a + b = {a + b}")
print(f"a - b = {a - b}")
print(f"a * b = {a * b}  (element-wise!)")
print(f"a / b = {a / b}")
print(f"a ** 2 = {a ** 2}")

# Operations with scalars (broadcasting!)
print(f"\na + 10 = {a + 10}  (adds 10 to each element)")
print(f"a * 2 = {a * 2}  (multiplies each element by 2)")

### Matrix Multiplication in NumPy

NumPy provides two ways to do matrix multiplication:
1. `np.matmul(A, B)` or `A @ B` (recommended)
2. `np.dot(A, B)` (older, but common)

**Documentation:**
- `np.matmul`: https://numpy.org/doc/stable/reference/generated/numpy.matmul.html
- `np.dot`: https://numpy.org/doc/stable/reference/generated/numpy.dot.html

## Summary

### What You've Learned

✅ **Computation graphs** track operations and their connections

✅ **Value class** records computation history for each operation

✅ **Forward propagation** computes outputs from inputs through the graph

✅ **Visualization** helps understand and debug computation graphs

✅ **Gradients** measure how sensitive outputs are to input changes

✅ **Numerical gradients** verify our understanding by nudging inputs

✅ **Chain rule** connects gradients through nested operations

✅ **Topological sort** enables reverse-mode automatic differentiation

✅ **Backward propagation** automatically computes all gradients

✅ **NumPy fundamentals** for efficient array operations

✅ **Matrix multiplication** with `@` operator and `np.matmul`

✅ **Linear regression with gradients** - combining Value objects with NumPy data

### Why This Matters

You've built a complete automatic differentiation system:
- ✓ Forward pass records operations
- ✓ Backward pass computes gradients automatically
- ✓ Value objects can be used with regular floats/NumPy
- ✓ Gradients have the same shape as parameters (shape [3] for 3 weights)

**This is exactly how PyTorch and TensorFlow work!**

### Next Steps

In **Lab 3**, you'll:
- Build neural network components (Neuron, Layer, MLP)
- Train a complete neural network on 2D data
- Visualize decision boundaries
- Solve the same problem as [Karpathy's micrograd demo](https://github.com/karpathy/micrograd/blob/master/demo.ipynb)

**Great work!** 🎉 You now have a working autograd engine!

In [ ]:
from utils import topological_sort

# Demo: Topological sort builds the correct order for backprop
a = Value(2.0, label='a')
b = Value(3.0, label='b')
c = Value(4.0, label='c')

e = a + b
e.label = 'e'
f = e * c
f.label = 'f'

# Get nodes in topological order (output to inputs)
topo = topological_sort(f)

print("Topological order (for backward pass):")
for node in topo:
    print(f"  {node.label if node.label else 'unnamed'}: data={node.data}")

print("\n✓ This is the order we traverse when computing gradients!")

### Adding Backward Pass to Value

To enable automatic differentiation, we need to add a `backward()` method to our Value class.

**Reference:** See [micrograd/engine.py](https://github.com/karpathy/micrograd/blob/master/micrograd/engine.py) for the complete implementation.

**Key concepts:**
1. Each operation stores a `_backward()` function
2. `backward()` method calls all `_backward()` functions in reverse topological order
3. Gradients accumulate (important when a node is used multiple times)

Here's the enhanced Value class with backward pass:

In [ ]:
# Complete Value class with backward pass (from micrograd)
class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None  # Default: no gradient computation
        self._prev = set(_children)
        self._op = _op
        self.label = label
    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        
        def _backward():
            # Gradient of addition: both inputs get the same gradient
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        
        return out
    
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        
        def _backward():
            # Gradient of multiplication: each input gets other's value times output gradient
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        
        return out
    
    def __pow__(self, other):
        assert isinstance(other, (int, float)), \"only supporting int/float powers\"
        out = Value(self.data ** other, (self,), f'**{other}')
        
        def _backward():
            # Gradient of power: other * x^(other-1) * out.grad
            self.grad += other * (self.data ** (other - 1)) * out.grad
        out._backward = _backward
        
        return out
    
    def backward(self):
        \"\"\"Compute gradients using reverse-mode automatic differentiation.\"\"\"
        # Build topological order
        topo = []
        visited = set()
        
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        
        build_topo(self)
        
        # Go backwards through graph
        self.grad = 1.0  # Gradient of output with respect to itself is 1
        for node in reversed(topo):
            node._backward()
    
    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other)
    def __truediv__(self, other): return self * (other ** -1)
    def __radd__(self, other): return self + other
    def __rsub__(self, other): return other + (-self)
    def __rmul__(self, other): return self * other
    
    def __repr__(self):
        return f\"Value(data={self.data:.4f}, grad={self.grad:.4f})\"\n\n# Test it!\na = Value(2.0, label='a')\nb = Value(3.0, label='b')\nc = a + b\nc.label = 'c'\n\nc.backward()  # Compute gradients!\n\nprint(\"After backward():\")\nprint(f\"a = {a}  (grad=1.0 because dc/da = 1)\")\nprint(f\"b = {b}  (grad=1.0 because dc/db = 1)\")\nprint(f\"c = {c}  (grad=1.0 because dc/dc = 1)\")"

## Exercise 3: 2D Linear Regression with Gradient Calculation

### Problem Setup

You have:
- **Input data**: 3 samples with 2 features each (as regular floats/numpy arrays)
- **Weights**: w0 (bias), w1, w2 (as Value objects - these are learnable!)
- **Model**: `y_pred = w0 + w1*x1 + w2*x2`
- **Target**: y_target (ground truth)
- **Loss**: Mean Squared Error (MSE)

### Your Task

1. Initialize weights as Value objects: `[w0, w1, w2]`
2. Compute predictions for all 3 samples using the linear model
3. Compute MSE loss
4. Call `backward()` to compute gradients
5. Extract gradients into a shape-[3] array

**Important:**
- Weights (w0, w1, w2) must be Value objects
- Input data (x1, x2) are regular floats
- Use NumPy for data handling
- Final gradient should be shape [3] matching the 3 weights

### Given Data

In [ ]:
import numpy as np

# Input data (3 samples, 2 features) - regular floats
X = np.array([
    [1.0, 2.0],   # Sample 1: x1=1.0, x2=2.0
    [2.0, 3.0],   # Sample 2: x1=2.0, x2=3.0
    [3.0, 4.0]    # Sample 3: x1=3.0, x2=4.0
])

# Target values - regular floats
y_target = np.array([5.0, 8.0, 11.0])

print(\"Data:\")\nprint(f\"X shape: {X.shape}\")\nprint(f\"X:\\n{X}\")\nprint(f\"\\ny_target shape: {y_target.shape}\")\nprint(f\"y_target: {y_target}\")"

### Your Implementation

Fill in the TODO sections below:

In [ ]:
# Solution: Linear Regression with Gradients

# Initialize weights as Value objects
w0 = Value(0.5, label='w0')
w1 = Value(0.3, label='w1')
w2 = Value(0.2, label='w2')

# Forward pass - compute predictions for all samples
predictions = []
for i in range(len(X)):
    y_pred = w0 + w1 * X[i, 0] + w2 * X[i, 1]
    predictions.append(y_pred)

# Compute MSE loss
loss = Value(0.0)
for i in range(len(X)):
    error = predictions[i] - y_target[i]
    loss = loss + error ** 2
loss = loss * (1.0 / len(X))  # Divide by 3

# Backward pass
loss.backward()

# Extract gradients
gradients = np.array([w0.grad, w1.grad, w2.grad])

print("\nWeights:")
print(f"w0 = {w0}")
print(f"w1 = {w1}")
print(f"w2 = {w2}")
print(f"\nLoss = {loss}")
print(f"\nGradients shape: {gradients.shape}  (should be (3,))")
print(f"Gradients: {gradients}")
print(f"\n✓ Gradients computed! Shape [3] matching [w0, w1, w2]")

### Hints

**1. Initialize weights:**
```python
w0 = Value(0.5, label='w0')  # Can start with any small value
```

**2. Forward pass example for one sample:**
```python
y_pred_0 = w0 + w1 * X[0, 0] + w2 * X[0, 1]
```

**3. Loss calculation:**
```python
# Accumulate squared errors
loss = Value(0.0)
for i in range(len(X)):
    y_pred = ...  # Your prediction
    error = y_pred - y_target[i]
    loss = loss + error ** 2
loss = loss * (1.0 / len(X))  # Divide by number of samples
```

**4. Remember:**
- Weights are Value objects
- Input data are floats
- Operations between Value and float work (we implemented `__rmul__`, etc.)

### Solution

In [ ]:
# Solution (uncomment to see)\n\n# # Initialize weights as Value objects\n# w0 = Value(0.5, label='w0')\n# w1 = Value(0.3, label='w1')\n# w2 = Value(0.2, label='w2')\n\n# # Forward pass - compute predictions for all samples\n# predictions = []\n# for i in range(len(X)):\n#     y_pred = w0 + w1 * X[i, 0] + w2 * X[i, 1]\n#     predictions.append(y_pred)\n\n# # Compute MSE loss\n# loss = Value(0.0)\n# for i in range(len(X)):\n#     error = predictions[i] - y_target[i]\n#     loss = loss + error ** 2\n# loss = loss * (1.0 / len(X))  # Divide by 3\n\n# # Backward pass\n# loss.backward()\n\n# # Extract gradients\n# gradients = np.array([w0.grad, w1.grad, w2.grad])\n\n# print(\"\\nWeights:\")\n# print(f\"w0 = {w0}\")\n# print(f\"w1 = {w1}\")\n# print(f\"w2 = {w2}\")\n# print(f\"\\nLoss = {loss}\")\n# print(f\"\\nGradients shape: {gradients.shape}\")\n# print(f\"Gradients: {gradients}\")\n# print(f\"\\n✓ Gradients computed! Shape [3] matching [w0, w1, w2]\")"

## Summary

### What You've Learned

✅ **Computation graphs** track operations and their connections

✅ **Value class** records computation history for each operation

✅ **Forward propagation** computes outputs from inputs through the graph

✅ **Visualization** helps understand and debug computation graphs

✅ **Gradients** measure how sensitive outputs are to input changes

✅ **Numerical gradients** verify our understanding by nudging inputs

✅ **Chain rule** connects gradients through nested operations

✅ **NumPy fundamentals** for efficient array operations

✅ **Matrix multiplication** with `@` operator and `np.matmul`

### Why This Matters

You've built the complete foundation for automatic differentiation:
- ✓ Forward pass records operations (this lab)
- ✓ Gradient concepts and chain rule (this lab)
- ⏳ Automatic backward pass (next lab)

### Next Steps

In **Lab 3**, you'll:
- Build neural network components (Neuron, Layer, MLP)
- Implement forward propagation through networks
- **Implement backward propagation** for each operator (manual autograd)
- Understand how gradients flow automatically through the graph

**Great work!** 🎉 You now understand:
- How deep learning frameworks track computations
- What gradients are and why they matter
- The foundation for automatic differentiation

## Exercise 2: Matrix Multiplication with NumPy

### Problem

You're building a simple neural network layer. Given:
- Input: `X` with shape (batch_size=3, input_features=4)
- Weights: `W` with shape (input_features=4, output_features=2)
- Bias: `b` with shape (output_features=2,)

Compute the output: `Y = X @ W + b`

### Your Task

1. Use NumPy to perform matrix multiplication
2. Add the bias (broadcasting will handle the shape)
3. Verify the output shape is (3, 2)

**Hint:** Review the `np.matmul` documentation if needed!

In [ ]:
import numpy as np

# Given data
X = np.array([[1, 2, 3, 4],
              [5, 6, 7, 8],
              [9, 10, 11, 12]])

W = np.array([[0.1, 0.2],
              [0.3, 0.4],
              [0.5, 0.6],
              [0.7, 0.8]])

b = np.array([1.0, 2.0])

print(f"X shape: {X.shape}  (3 samples, 4 features each)")
print(f"W shape: {W.shape}  (4 input features, 2 output features)")
print(f"b shape: {b.shape}  (2 output features)\n")

# Solution: Matrix multiplication with NumPy
Y = X @ W + b

print(f"Y shape: {Y.shape}  (should be (3, 2))")
print(f"Y:\n{Y}")

# Verify
expected_shape = (3, 2)
assert Y.shape == expected_shape, f"Expected shape {expected_shape}, got {Y.shape}"
print("\n✓ Matrix multiplication: PASS")

### Solution

In [ ]:
# Solution (uncomment to see)
# Y = X @ W + b
# 
# Or equivalently:
# Y = np.matmul(X, W) + b
# 
# The @ operator is cleaner and recommended!

## Summary

### What You've Learned

✅ **Computation graphs** track operations and their connections

✅ **Value class** records computation history for each operation

✅ **Forward propagation** computes outputs from inputs through the graph

✅ **Visualization** helps understand and debug computation graphs

✅ **NumPy fundamentals** for efficient array operations

✅ **Matrix multiplication** with `@` operator and `np.matmul`

### Why This Matters

You've built the foundation for automatic differentiation:
- ✓ Forward pass records operations (this lab)
- ⏳ Backward pass computes gradients (next lab)

### Next Steps

In **Lab 3**, you'll:
- Build neural network components (Neuron, Layer, MLP)
- Implement forward propagation through networks
- Implement backward propagation manually
- Understand how gradients flow through the graph

**Great work!** 🎉 You now understand how deep learning frameworks track computations.